In [1]:
# Importações e Inicializacao da SparkSession
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Formatos-Spark-Parquet")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-avro_2.12:3.5.0")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "root-minio")
    .config("spark.hadoop.fs.s3a.secret.key", "root12345678")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

print(f"SparkSession ativa. Versao: {spark.version}")

SparkSession ativa. Versao: 3.5.0


In [2]:
# Extracao (Extract) do PostgreSQL via JDBC
print("Extraindo transacoes financeiras para processamento colunar...")

df_financeiro = (spark.read.jdbc(
    url="jdbc:postgresql://postgres:5432/postgres",
    table="transacoes_financeiras",
    properties={"user": "postgres", "password": "senha123", "driver": "org.postgresql.Driver"}
))

# O Spark resolve nativamente os tipos UUID e mapeia corretamente para a estrutura interna
df_financeiro.printSchema()

Extraindo transacoes financeiras para processamento colunar...
root
 |-- id_transacao: string (nullable = true)
 |-- conta_origem: string (nullable = true)
 |-- conta_destino: string (nullable = true)
 |-- tipo_movimento: string (nullable = true)
 |-- valor: decimal(15,4) (nullable = true)
 |-- moeda: string (nullable = true)
 |-- data_transacao: timestamp (nullable = true)
 |-- hash_auditoria: string (nullable = true)
 |-- data_criacao: timestamp (nullable = true)



In [4]:
# Carga (Load) para o MinIO em formato Parquet
bucket_name = "raw"
destino_s3 = f"s3a://{bucket_name}/spark_jobs/ledger_parquet/"

print(f"Gravando dados em formato Parquet com compressao Snappy: {destino_s3}")

# Por padrao, o Spark aplica a compressao Snappy ao persistir arquivos Parquet
df_financeiro.write.mode("overwrite").parquet(destino_s3)

print("Escrita do Parquet concluida com sucesso no destino.")

Gravando dados em formato Parquet com compressao Snappy: s3a://raw/spark_jobs/ledger_parquet/
Escrita do Parquet concluida com sucesso no destino.


In [5]:
# Auditoria e Validação dos Dados Gravados
print("Lendo arquivos Parquet de volta do Data Lake para validacao...")

df_validacao = spark.read.parquet(destino_s3)

print(f"Quantidade total de registros localizados: {df_validacao.count()}")
df_validacao.show(5, truncate=False)

Lendo arquivos Parquet de volta do Data Lake para validacao...
Quantidade total de registros localizados: 20000
+------------------------------------+------------------+------------------+--------------+----------+-----+-------------------+----------------------------------------------------------------+-------------------+
|id_transacao                        |conta_origem      |conta_destino     |tipo_movimento|valor     |moeda|data_transacao     |hash_auditoria                                                  |data_criacao       |
+------------------------------------+------------------+------------------+--------------+----------+-----+-------------------+----------------------------------------------------------------+-------------------+
|e9b4ba43-ef07-4d76-8436-4c829966c49b|IYXG33090884540679|NMIZ81024723375965|DEBITO        |14236.8874|BRL  |2026-04-16 23:53:15|e23c1a3adb95d63fdc32c8eedd0f87db6fd050073b12e7359e4af8d17677ab40|2026-04-16 23:53:15|
|000df2fb-d92d-4bef-b71c-6d9094a

In [6]:
# Encerramento da Sessão e Limpeza de Recursos
print("Encerrando a SparkSession e liberando os recursos da JVM...")
spark.stop()
print("Sessao fechada com seguranca.")

Encerrando a SparkSession e liberando os recursos da JVM...
Sessao fechada com seguranca.


### Análise de Arquitetura: O Formato Parquet no Apache Spark

**Características Principais:** Formato binário, autodescritivo e estritamente orientado a colunas (Columnar). É o formato nativo e padrão ouro do Apache Spark.

#### Vantagens
* **Compressão Eficiente:** Como dados do mesmo tipo ficam armazenados sequencialmente lado a lado (coluna após coluna), algoritmos de compressão como o Snappy operam com eficiência máxima, reduzindo o tamanho de armazenamento em até 75% comparado ao CSV.
* **Column Projection:** Se uma tabela possui 100 colunas e uma consulta SQL solicita apenas duas, o motor analítico lê exclusivamente os bytes dessas duas colunas no storage, ignorando os outros 98%. Isso zera o desperdício de I/O de rede.
* **Predicate Pushdown:** O rodapé (footer) de cada arquivo Parquet armazena estatísticas como valores mínimos e máximos de cada bloco. Se uma query filtra por uma data específica, o Spark lê o footer e pula blocos inteiros de arquivos que não contêm aquele intervalo, sem precisar abrir os dados.

#### Desvantagens
* **Incompatível com Operações Transacionais Unitárias:** Não foi projetado para suportar atualizações ou inserções de linhas únicas (Inserts/Updates na granularidade de um registro). Modificar uma linha exige reescrever o bloco colunar inteiro.
* **Ilegibilidade:** Por ser um formato binário compactado, não pode ser aberto ou lido por humanos sem o auxílio de ferramentas de query ou código.

#### Casos de Uso no Mercado
* Formato padrão para o armazenamento de dados analíticos de alta performance (camadas Silver e Gold de Data Lakes e Lakehouses).
* Formato de persistência oficial para motores de consulta OLAP baseados em nuvem, como AWS Athena, Databricks, Google BigQuery e Apache Hive.